In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.runtime import load_defaults, load_predictor
from src.modeling.phases import (
    probabilities_to_calibrated_labels,
    probabilities_to_relaxed_labels,
)

In [ ]:
RUN_DIR = ""  # latest diffusion run
BATCH_SIZE = 16

In [ ]:
run_dir = Path(RUN_DIR) if RUN_DIR else max(
    [p for p in (ROOT / "run").glob("*") if (p / "diffusion.yaml").is_file()],
    key=lambda p: p.stat().st_mtime,
)
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_phases = load_defaults(run_dir / "vae.yaml")["num_phases"]
predictor = load_predictor(run_dir, run_dir, device=device)
vae = predictor.vae

print("run:", run_dir.name)

In [ ]:
shape = (
    BATCH_SIZE,
    int(vae.latent_ch),
    int(vae.latent_size),
    int(vae.latent_size),
)
with torch.no_grad():
    latent = predictor.sampler.sample(shape)
    probabilities = vae.decode_probs(latent).detach().cpu()

if not torch.isfinite(probabilities).all():
    raise RuntimeError("Sampling diverged: decoded probabilities contain NaN or Inf.")

generated = probabilities_to_relaxed_labels(probabilities, num_phases)
phase = probabilities_to_calibrated_labels(probabilities, num_phases)
phase_fractions = [
    (phase == value).float().mean().item() for value in range(num_phases)
]
print("phase fractions:", [round(value, 4) for value in phase_fractions])

items = [("generated", generated[0, 0]), ("phase", phase[0, 0])]
fig, axes = plt.subplots(1, 2, figsize=(8, 4), squeeze=False)
for ax, (title, image) in zip(axes.ravel(), items):
    ax.imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=num_phases - 1,
        interpolation="nearest",
    )
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()